# CNN-Transformer-BiGRU v7（彻底回退版）

修复策略：完全复刻原模型成功架构（the_other_model.ipynb），仅做最小改动
- 完全回退到原模型的 Post-Norm Transformer
- 完全回退到原模型的双路相加融合
- 完全回退到原模型的 Dropout 0.1
- 仅添加：输出层前的 Dropout（防止最终层过拟合）

原模型已验证：MAPE ~3%，Train/Val Loss 差距小

In [ ]:
import math
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)

---
## 第 1 步：配置（完全复刻原模型）

In [ ]:
@dataclass
class ModelConfig:
    input_dim: int = 53
    cnn_out_dim: int = 32
    d_model: int = 128
    nhead: int = 4
    num_encoder_layers: int = 2
    dim_feedforward: int = 256
    bigru_hidden: int = 64
    bigru_layers: int = 1
    output_len: int = 24
    dropout: float = 0.1            # 完全复刻原模型

config = ModelConfig()
print(config)

---
## 第 2 步：位置编码（完全复刻）

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

pe = PositionalEncoding(config.d_model)
x = torch.randn(2, 168, config.d_model)
assert pe(x).shape == x.shape
print("PositionalEncoding OK")

---
## 第 3 步：多头注意力（完全复刻）

In [ ]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model = d_model
        self.nhead = nhead
        self.d_k = d_model // nhead
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.w_q(q).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        K = self.w_k(k).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        V = self.w_v(v).view(B, -1, self.nhead, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.w_o(out), attn

attn = MultiheadAttention(config.d_model, config.nhead)
x = torch.randn(2, 168, config.d_model)
out, w = attn(x, x, x)
assert out.shape == (2, 168, config.d_model) and w.shape == (2, 4, 168, 168)
print("MultiheadAttention OK")

---
## 第 4 步：Post-Norm TransformerEncoderLayer（完全复刻原模型）

In [ ]:
class TransformerEncoderLayer(nn.Module):
    """Post-Norm：完全复刻原模型成功架构"""
    def __init__(self, d_model, nhead, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, mask=None):
        src2, attn = self.self_attn(src, src, src, mask)
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = self.norm2(src + self.dropout2(src2))
        return src, attn

layer = TransformerEncoderLayer(config.d_model, config.nhead)
x = torch.randn(2, 168, config.d_model)
out, _ = layer(x)
assert out.shape == x.shape
print("TransformerEncoderLayer (Post-Norm) OK")

---
## 第 5 步：TransformerEncoder（完全复刻）

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, nhead, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, src, mask=None):
        attn_list = []
        for layer in self.layers:
            src, attn = layer(src, mask)
            attn_list.append(attn)
        return src, attn_list

encoder = TransformerEncoder(config.num_encoder_layers, config.d_model, config.nhead)
x = torch.randn(2, 168, config.d_model)
out, al = encoder(x)
assert len(al) == config.num_encoder_layers
print(f"TransformerEncoder OK (layers={config.num_encoder_layers})")

---
## 第 6 步：CNN（完全复刻原模型）

In [ ]:
class EdgeDetectCNN(nn.Module):
    def __init__(self, in_ch, out_ch=32, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch // 2, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_ch // 2)
        self.conv2 = nn.Conv1d(out_ch // 2, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.shortcut = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.transpose(1, 2)
        s = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out = self.relu(out + s)
        out = self.dropout(out)
        return out.transpose(1, 2)

cnn = EdgeDetectCNN(53, 32)
x = torch.randn(4, 168, 53)
assert cnn(x).shape == (4, 168, 32)
print("EdgeDetectCNN OK")

---
## 第 7 步：主模型 v7（完全复刻原模型 + 仅添加输出前 Dropout）

In [ ]:
class Improved_CNN_Transformer_BiGRU(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.cnn = EdgeDetectCNN(config.input_dim, config.cnn_out_dim, config.dropout)
        
        # 完全复刻原模型：双路相加融合
        self.raw_projection = nn.Linear(config.input_dim, config.d_model)
        self.cnn_projection = nn.Linear(config.cnn_out_dim, config.d_model)
        
        self.pos_encoder = PositionalEncoding(config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        
        # Transformer 编码器（完全复刻原模型：2层 Post-Norm）
        self.transformer_encoder = TransformerEncoder(
            config.num_encoder_layers, config.d_model, config.nhead,
            config.dim_feedforward, config.dropout
        )
        self.transformer_norm = nn.LayerNorm(config.d_model)
        
        # BiGRU（完全复刻）
        self.bigru = nn.GRU(
            input_size=config.d_model,
            hidden_size=config.bigru_hidden,
            num_layers=config.bigru_layers,
            batch_first=True, bidirectional=True
        )
        
        # 输出层（仅添加输出前 Dropout，防止最终层过拟合）
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.pre_output_dropout = nn.Dropout(config.dropout)  # 唯一改动
        self.output_layer = nn.Linear(config.bigru_hidden * 2, config.output_len)
        
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        # x: (B, 168, 53)
        cnn_out = self.cnn(x)                    # (B, 168, 32)
        
        # 双路融合：相加（完全复刻原模型）
        raw_feat = self.raw_projection(x)        # (B, 168, 128)
        cnn_feat = self.cnn_projection(cnn_out)  # (B, 168, 128)
        fused = raw_feat + cnn_feat              # (B, 168, 128)
        
        x = self.pos_encoder(fused)
        x = self.dropout(x)
        
        # Transformer（2层 Post-Norm，完全复刻）
        x, attn_list = self.transformer_encoder(x)  # (B, 168, 128)
        x = self.transformer_norm(x)
        
        # BiGRU
        x, _ = self.bigru(x)                     # (B, 168, 128)
        
        # 全局池化 + Dropout + 输出
        x = self.global_pool(x.transpose(1, 2)).squeeze(-1)  # (B, 128)
        x = self.pre_output_dropout(x)           # 唯一改动：防止最终层过拟合
        out = self.output_layer(x)               # (B, 24)
        return out, attn_list

model = Improved_CNN_Transformer_BiGRU(config)
x = torch.randn(4, 168, 53)
out, attn = model(x)
assert out.shape == (4, 24)
print(f"Model OK: {x.shape} -> {out.shape}, attn={len(attn)}")

---
## 第 8 步：参数量

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total:,} ({total * 4 / 1024 / 1024:.2f} MB)")

---
## 第 9 步：消融基线 CNN_BiGRU

In [ ]:
class CNN_BiGRU(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.cnn = EdgeDetectCNN(config.input_dim, config.cnn_out_dim, config.dropout)
        self.bigru = nn.GRU(
            input_size=config.cnn_out_dim,
            hidden_size=config.bigru_hidden,
            num_layers=config.bigru_layers,
            batch_first=True, bidirectional=True
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.output_layer = nn.Linear(config.bigru_hidden * 2, config.output_len)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        cnn_out = self.cnn(x)
        x_bigru, _ = self.bigru(cnn_out)
        x_pool = self.global_pool(x_bigru.transpose(1, 2)).squeeze(-1)
        return self.output_layer(x_pool)

baseline = CNN_BiGRU(config)
x = torch.randn(4, 168, 53)
assert baseline(x).shape == (4, 24)
total_b = sum(p.numel() for p in baseline.parameters())
print(f"Baseline OK: params={total_b:,}, full={total:,}, diff={total-total_b:,}")